In [6]:
import os
import glob
from PIL import Image

# ----------------------------
# Step 1: Data Preparation
# ----------------------------

# Define paths
input_dir = '/kaggle/input/endoscopygoodimages/Original'
resized_dir = '/kaggle/working/resized_images'
patch_dir = '/kaggle/working/image_patches'

# Create directories if they don't exist
os.makedirs(resized_dir, exist_ok=True)
os.makedirs(patch_dir, exist_ok=True)

# Set target resolution
target_size = (299, 299)

def resize_images(input_dir, output_dir, target_size):
    """
    Resize all images in input_dir to target_size and save them to output_dir.
    Uses bicubic interpolation to preserve details.
    """
    image_files = glob.glob(os.path.join(input_dir, '*'))
    for img_path in image_files:
        try:
            with Image.open(img_path) as img:
                # Convert image to RGB (if not already) for consistency
                img = img.convert('RGB')
                # Resize the image using bicubic interpolation
                resized_img = img.resize(target_size, Image.BICUBIC)
                base_name = os.path.basename(img_path)
                output_path = os.path.join(output_dir, base_name)
                resized_img.save(output_path)
                print(f"Saved resized image: {output_path}")
        except Exception as e:
            print(f"Error processing {img_path}: {e}")

# Run the resizing function
resize_images(input_dir, resized_dir, target_size)

def extract_patches(image, patch_size, overlap):
    """
    Extract overlapping patches from an image.
    :param image: PIL Image object.
    :param patch_size: Size of the square patch.
    :param overlap: Overlap in pixels between patches.
    :return: List of tuples (patch, left, top) where left and top are the patch coordinates.
    """
    width, height = image.size
    patches = []
    step = patch_size - overlap
    for top in range(0, height - patch_size + 1, step):
        for left in range(0, width - patch_size + 1, step):
            patch = image.crop((left, top, left + patch_size, top + patch_size))
            patches.append((patch, left, top))
    return patches

# Parameters for patch extraction (optional)
patch_size = 128
overlap = 16

def create_patches_from_resized(resized_dir, patch_dir, patch_size, overlap):
    """
    Extract patches from each resized image and save them.
    """
    image_files = glob.glob(os.path.join(resized_dir, '*'))
    for img_path in image_files:
        try:
            with Image.open(img_path) as img:
                img = img.convert('RGB')
                patches = extract_patches(img, patch_size, overlap)
                base_name = os.path.splitext(os.path.basename(img_path))[0]
                for idx, (patch, left, top) in enumerate(patches):
                    patch_filename = f"{base_name}_patch_{left}_{top}.png"
                    patch.save(os.path.join(patch_dir, patch_filename))
                print(f"Extracted {len(patches)} patches from {img_path}")
        except Exception as e:
            print(f"Error extracting patches from {img_path}: {e}")

# Uncomment the next line to run patch extraction if needed
create_patches_from_resized(resized_dir, patch_dir, patch_size, overlap)

Saved resized image: /kaggle/working/resized_images/663afdb6-38bb-450c-9f53-a5273c46ea77.jpg
Saved resized image: /kaggle/working/resized_images/f95bc1b1-9da1-4970-a3ac-c11155b177e6.jpg
Saved resized image: /kaggle/working/resized_images/cd39f3fa-91eb-4798-bf8e-d9ccfaf1bbcd.jpg
Saved resized image: /kaggle/working/resized_images/b96a8420-729a-42a5-a133-dc41e72f84be.jpg
Saved resized image: /kaggle/working/resized_images/b32c2284-32ff-4328-9edc-355a96d6c9e3.jpg
Saved resized image: /kaggle/working/resized_images/00fb1871-41c0-4619-be56-29690d145b1d.jpg
Saved resized image: /kaggle/working/resized_images/7ffe7230-fb4b-41de-9929-3f036a08d5dd.jpg
Saved resized image: /kaggle/working/resized_images/da85aa6e-ba2f-416c-a6d2-e93e1db2b6d3.jpg
Saved resized image: /kaggle/working/resized_images/341880bd-c20e-4715-bd10-d0c6ff460988.jpg
Saved resized image: /kaggle/working/resized_images/00091.jpg
Saved resized image: /kaggle/working/resized_images/2b4c6723-620d-4a70-a3ab-d9e8697f4df2.jpg
Saved re

### SRCNN

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SRCNN(nn.Module):
    def __init__(self):
        super(SRCNN, self).__init__()
        # First layer: large kernel to capture context
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=64, kernel_size=9, padding=4)
        # Second layer: refine features
        self.conv2 = nn.Conv2d(in_channels=64, out_channels=32, kernel_size=5, padding=2)
        # Third layer: reconstruct output image with 3 channels
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=3, kernel_size=5, padding=2)
    
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.conv3(x)
        return x

# Testing SRCNN forward pass (for sanity check)
if __name__ == '__main__':
    srcnn_model = SRCNN()
    dummy_input = torch.randn(1, 3, 299, 299)  # one sample, RGB, 299x299
    output = srcnn_model(dummy_input)
    print("SRCNN output shape:", output.shape)

SRCNN output shape: torch.Size([1, 3, 299, 299])


### Dual Input U-Net for Enhancement

In [9]:
class UNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UNetBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        return x

def center_crop(tensor, target_size):
    """
    Center crop a tensor to target_size (height, width).
    Assumes tensor shape is (B, C, H, W).
    """
    _, _, h, w = tensor.size()
    target_h, target_w = target_size
    delta_h = h - target_h
    delta_w = w - target_w
    top = delta_h // 2
    left = delta_w // 2
    return tensor[:, :, top:top+target_h, left:left+target_w]

class DualInputUNet(nn.Module):
    def __init__(self, input_channels=6, output_channels=3):
        super(DualInputUNet, self).__init__()
        # Encoder
        self.enc1 = UNetBlock(input_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = UNetBlock(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        
        # Bottleneck
        self.bottleneck = UNetBlock(128, 256)
        
        # Decoder
        self.up1 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec1 = UNetBlock(256, 128)  # Concatenation of up1 and cropped enc2 (128 + 128 = 256)
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = UNetBlock(128, 64)   # Concatenation of up2 and cropped enc1 (64 + 64 = 128)
        
        # Final output
        self.final_conv = nn.Conv2d(64, output_channels, kernel_size=1)
    
    def forward(self, orig, super_res):
        # Concatenate original and super-resolved images along the channel dimension.
        # Expected input shape for each: (B, 3, H, W), output shape: (B, 6, H, W)
        x = torch.cat([orig, super_res], dim=1)
        
        # Encoder
        enc1 = self.enc1(x)             # -> (B, 64, H, W)
        enc2 = self.enc2(self.pool1(enc1))# -> (B, 128, H/2, W/2)
        
        # Bottleneck
        bottleneck = self.bottleneck(self.pool2(enc2))  # -> (B, 256, H/4, W/4)
        
        # Decoder
        up1 = self.up1(bottleneck)      # -> (B, 128, H/2, W/2)
        # Crop enc2 to match the dimensions of up1 before concatenation
        enc2_cropped = center_crop(enc2, up1.shape[2:])
        dec1 = self.dec1(torch.cat([up1, enc2_cropped], dim=1))
        
        up2 = self.up2(dec1)            # -> (B, 64, H, W)
        # Crop enc1 to match the dimensions of up2 before concatenation
        enc1_cropped = center_crop(enc1, up2.shape[2:])
        dec2 = self.dec2(torch.cat([up2, enc1_cropped], dim=1))
        
        # Final convolution to produce the enhanced image
        out = self.final_conv(dec2)
        return out

In [10]:
if __name__ == '__main__':
    unet_model = DualInputUNet()
    orig_dummy = torch.randn(1, 3, 299, 299)
    super_res_dummy = torch.randn(1, 3, 299, 299)
    enhanced_output = unet_model(orig_dummy, super_res_dummy)
    print("Dual-Input U-Net enhanced output shape:", enhanced_output.shape)

Dual-Input U-Net enhanced output shape: torch.Size([1, 3, 296, 296])


### Custom Dataset and Loader

In [16]:
import os
import glob
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# ----------------------------
# Custom Dataset (from Part 1)
# ----------------------------
class CustomDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        # Update glob pattern for JPG images
        self.image_paths = glob.glob(os.path.join(image_dir, '*.jpg'))
        print(f"Found {len(self.image_paths)} images in {image_dir}")
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        # For this demo, we use the same image as low-res input and high-res ground truth.
        return {'low_res': image, 'high_res': image}

# ----------------------------
# Image Transformations
# ----------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
])

# Set the directory for resized images
resized_dir = '/kaggle/working/resized_images'
train_dataset = CustomDataset(resized_dir, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

# ----------------------------
# Custom Loss Functions (from Part 2)
# ----------------------------
def edge_loss(output, target):
    """
    Compute an edge-aware loss using a Sobel filter.
    Measures the L1 difference between gradients of the output and target.
    """
    sobel_kernel = torch.tensor([[[[-1, 0, 1],
                                   [-2, 0, 2],
                                   [-1, 0, 1]]]], dtype=torch.float32, device=output.device)
    channels = output.size(1)
    sobel_kernel = sobel_kernel.repeat(channels, 1, 1, 1)
    
    grad_output = F.conv2d(output, sobel_kernel, padding=1, groups=channels)
    grad_target = F.conv2d(target, sobel_kernel, padding=1, groups=channels)
    
    return F.l1_loss(grad_output, grad_target)

def combined_loss(output, target, alpha=1.0, beta=1.0, gamma=1.0):
    """
    Combine L1, MSE, and edge-aware losses.
    Adjust alpha, beta, and gamma to emphasize different components.
    """
    loss_l1 = F.l1_loss(output, target)
    loss_mse = F.mse_loss(output, target)
    loss_edge = edge_loss(output, target)
    return alpha * loss_l1 + beta * loss_mse + gamma * loss_edge

# ----------------------------
# Model Definitions (from Step 2)
# ----------------------------
class SRCNN(nn.Module):
    def __init__(self):
        super(SRCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=9, padding=4)
        self.conv2 = nn.Conv2d(64, 32, kernel_size=5, padding=2)
        self.conv3 = nn.Conv2d(32, 3, kernel_size=5, padding=2)
    
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.conv3(x)
        return x

class UNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UNetBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        return x

def center_crop(tensor, target_size):
    """
    Center crop a tensor to target_size (height, width).
    Assumes tensor shape is (B, C, H, W).
    """
    _, _, h, w = tensor.size()
    target_h, target_w = target_size
    delta_h = h - target_h
    delta_w = w - target_w
    top = delta_h // 2
    left = delta_w // 2
    return tensor[:, :, top:top+target_h, left:left+target_w]

class DualInputUNet(nn.Module):
    def __init__(self, input_channels=6, output_channels=3):
        super(DualInputUNet, self).__init__()
        # Encoder
        self.enc1 = UNetBlock(input_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = UNetBlock(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        
        # Bottleneck
        self.bottleneck = UNetBlock(128, 256)
        
        # Decoder
        self.up1 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec1 = UNetBlock(256, 128)  # up1 (128) + cropped enc2 (128)
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = UNetBlock(128, 64)   # up2 (64) + cropped enc1 (64)
        
        # Final output
        self.final_conv = nn.Conv2d(64, output_channels, kernel_size=1)
    
    def forward(self, orig, super_res):
        # Concatenate original and super-resolved images: shape (B,6,H,W)
        x = torch.cat([orig, super_res], dim=1)
        
        # Encoder
        enc1 = self.enc1(x)                     # -> (B,64,H,W)
        enc2 = self.enc2(self.pool1(enc1))        # -> (B,128,H/2,W/2)
        
        # Bottleneck
        bottleneck = self.bottleneck(self.pool2(enc2))  # -> (B,256,H/4,W/4)
        
        # Decoder
        up1 = self.up1(bottleneck)              # -> (B,128,H/2,W/2)
        enc2_cropped = center_crop(enc2, up1.shape[2:])
        dec1 = self.dec1(torch.cat([up1, enc2_cropped], dim=1))
        
        up2 = self.up2(dec1)                    # -> (B,64,H,W)
        enc1_cropped = center_crop(enc1, up2.shape[2:])
        dec2 = self.dec2(torch.cat([up2, enc1_cropped], dim=1))
        
        out = self.final_conv(dec2)
        return out

# ----------------------------
# Setup Training Environment
# ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
srcnn_model = SRCNN().to(device)
unet_model = DualInputUNet().to(device)

# Combine model parameters for the optimizer
optimizer = optim.Adam(list(srcnn_model.parameters()) + list(unet_model.parameters()), lr=1e-4)

# ----------------------------
# Training Loop
# ----------------------------
num_epochs = 10  # Adjust as needed

for epoch in range(num_epochs):
    srcnn_model.train()
    unet_model.train()
    running_loss = 0.0
    
    for batch in train_loader:
        low_res = batch['low_res'].to(device)   # shape: [B,3,299,299]
        high_res = batch['high_res'].to(device) # shape: [B,3,299,299]
        
        optimizer.zero_grad()
        
        # Generate super-resolved image using SRCNN
        super_res = srcnn_model(low_res)
        
        # Generate enhanced image using Dual-Input U-Net
        enhanced = unet_model(low_res, super_res)
        
        # Because the U-Net output may be slightly smaller due to cropping,
        # center crop the high-res target to match the enhanced output dimensions.
        _, _, H, W = enhanced.size()
        high_res_cropped = transforms.functional.center_crop(high_res, [H, W])
        
        # Compute the combined loss
        loss = combined_loss(enhanced, high_res_cropped)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

Found 1887 images in /kaggle/working/resized_images
Epoch 1/10, Loss: 0.1081
Epoch 2/10, Loss: 0.0275
Epoch 3/10, Loss: 0.0195
Epoch 4/10, Loss: 0.0168
Epoch 5/10, Loss: 0.0148
Epoch 6/10, Loss: 0.0135
Epoch 7/10, Loss: 0.0116
Epoch 8/10, Loss: 0.0110
Epoch 9/10, Loss: 0.0104
Epoch 10/10, Loss: 0.0098


In [18]:
import torchvision.transforms.functional as TF

def infer_full_image(image_path, srcnn_model, unet_model, transform, device):
    # Load and transform image
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(device)  # shape: [1,3,H,W]
    
    srcnn_model.eval()
    unet_model.eval()
    with torch.no_grad():
        # Generate super-resolved image
        super_res = srcnn_model(image_tensor)
        # Generate enhanced image
        enhanced = unet_model(image_tensor, super_res)
    
    # Optionally, center crop the input high-res if necessary (if sizes don't match)
    _, _, H, W = enhanced.size()
    image_tensor_cropped = TF.center_crop(image_tensor, [H, W])
    
    # Convert enhanced output to PIL image for visualization
    enhanced_image = TF.to_pil_image(enhanced.squeeze(0).cpu())
    return enhanced_image

# Example usage:
test_image_path = "/kaggle/working/resized_images/00004_batch2.jpg"
enhanced_img = infer_full_image(test_image_path, srcnn_model, unet_model, transform, device)
enhanced_img.show()  # or save using enhanced_img.save('enhanced_sample.jpg')